# Fake Jobs – Enhanced (PCA 30)
- PCA über den TabPFN-Embedding-Block, **nur auf Trainingszeilen gefittet**
- Erklärte Varianz wird geprintet (Ziel > 80 %)

In [ ]:
import os
import pandas as pd
from sklearn.decomposition import PCA

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Enhanced-CSV & Split laden

In [ ]:
df = pd.read_csv(f"../../data/preprocessed/enhanced_fake_jobs_seed{SEED}.csv")
split = pd.read_csv(f"../../data/splits/split_fake_jobs_seed{SEED}.csv")
emb_cols = [c for c in df.columns if c.startswith("emb_")]
tr = df["row_id"].isin(split.loc[split["split"] == "train", "row_id"]).values
print("Embedding-Spalten:", len(emb_cols), "| Train:", int(tr.sum()))

## PCA(30) – auf Train gefittet, speichern

In [ ]:
pca = PCA(n_components=30, random_state=SEED).fit(df.loc[tr, emb_cols])
red = pca.transform(df[emb_cols])
print("explained variance (30 comps):", round(pca.explained_variance_ratio_.sum(), 4))

out = pd.concat([df[["row_id"]].reset_index(drop=True),
                 pd.DataFrame(red, columns=[f"pca_{i}" for i in range(30)])], axis=1)
out["fraudulent"] = df["fraudulent"].values
out.to_csv(f"../../data/preprocessed/enhanced_pca30_fake_jobs_seed{SEED}.csv", index=False)
print("Shape:", out.shape)

## Verifikation

In [ ]:
assert out.isna().sum().sum() == 0
assert out["row_id"].is_unique